# **Libraries**

In [ ]:
# --------------------------------------------------------
#  Import Libraries
# --------------------------------------------------------
import base64, json, time, msal, requests, pyodbc
from typing import Dict, List, Optional
import notebookutils.credentials as cred

# **Azure Key Vault**

In [ ]:
# --------------------------------------------------------
# Define Azure Key Vault
# --------------------------------------------------------
key_vault_name = "<KeyVaultName>"
key_vault_url = f"https://{key_vault_name}.vault.azure.net/"

# --------------------------------------------------------
# Get Azure Key Vault Service Principal Credentials
# --------------------------------------------------------
client_name = "<ServicePrincipalName>"
tenant_id = cred.getSecret(key_vault_url,"<ServicePrincipalTenantIdSecretName>")
client_id = cred.getSecret(key_vault_url,"<ServicePrincipalClientIdSecretName>")
client_secret = cred.getSecret(key_vault_url,"<ServicePrincipalClientSecretSecretName>")

# **Functions**

## **Access token function**

In [ ]:
# --------------------------------------------------------
#  Get access token function
# --------------------------------------------------------
def get_access_token(tenant_id, client_id, client_secret):
    """
    Retrieve a Fabric API access token using a service principal.

    :param tenant_id: Entra ID tenant GUID.
    :param client_id: Service principal application (client) ID.
    :param client_secret: Service principal client secret.
    :returns: OAuth2 bearer token string (audience: api.fabric.microsoft.com).
    :raises RuntimeError: If the token request fails or returns no token.
    """

    # token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

    headers = {'Content-Type': 'application/x-www-form-urlencoded'}

    body = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'resource': 'https://api.fabric.microsoft.com',
        'scope': 'https://api.fabric.microsoft.com/.default'
    }

    try:
        response = requests.post(token_provider_uri, data=body, headers=headers)
        response.raise_for_status()
    except requests.exceptions.HTTPError as http_err:
        # Fabric loves giving vague 401s, so let's make it obvious
        raise RuntimeError(
            f"Failed to retrieve access token. "
            f"HTTP error: {http_err}. "
            f"Check tenant ID, client ID, secret, and SP permissions."
        )
    except Exception as ex:
        raise RuntimeError(
            f"Unexpected error while retrieving access token: {ex}"
        )

    token = response.json().get("access_token")
    if not token:
        raise RuntimeError("Token request succeeded but no access_token was returned.")

    print("Access token successfully retrieved.")
    return token

# --------------------------------------------------------
#  Long-running operation poller
# --------------------------------------------------------
def poll_lro(
        response: requests.Response
      , headers: Dict
      , return_result: bool = False
      , poll_timeout_seconds: int = 300
) -> requests.Response:
    """
    Poll a Fabric long-running operation (LRO) until completion.

    Fabric mutating APIs (item creation, definition updates, etc.) may return
    202 Accepted with a Location header pointing at the operation status
    endpoint. This helper polls that endpoint, honoring Retry-After, until the
    operation reaches a terminal state.

    When the caller receives a non-202 response (e.g. 201 Created with the new
    item's body), this function returns it unchanged — so callers can safely
    wrap every `requests.post(...)` regardless of the response shape.

    :param response: The initial HTTP response from the POST/PATCH call.
    :param headers: Headers dict to reuse for polling (must include Authorization).
    :param return_result: When True, after a successful status, GET the
        operation's `/result` endpoint and return that response. Useful when
        the caller needs the created item's body (e.g. to extract its ID).
    :param poll_timeout_seconds: Maximum seconds to poll before giving up.
    :returns: Final requests.Response — either the original (if not 202) or the
        terminal operation status response, or the `/result` body when
        return_result=True.
    :raises RuntimeError: If the operation completes with a non-Succeeded status.
    :raises TimeoutError: If the operation does not reach a terminal state
        within poll_timeout_seconds.
    """

    # Caller got a synchronous response — pass through unchanged
    if response.status_code != 202:
        return response

    location = response.headers.get("Location")
    if not location:
        raise RuntimeError(
            "Received 202 but no Location header for LRO polling."
        )

    deadline = time.time() + poll_timeout_seconds

    # Poll until the operation reaches a terminal state (Succeeded, Failed, etc.)
    while time.time() < deadline:
        retry_after = int(response.headers.get("Retry-After", 5))
        print(f"  LRO in progress, retrying in {retry_after}s...")
        time.sleep(retry_after)

        response = requests.get(location, headers=headers)
        status = response.json().get("status", "")

        if status not in ("NotStarted", "Running"):
            break
    else:
        raise TimeoutError(
            f"LRO did not complete within {poll_timeout_seconds}s: {location}"
        )

    final_status = response.json().get("status")
    if final_status != "Succeeded":
        raise RuntimeError(
            f"LRO ended with non-success status '{final_status}'. "
            f"Response: {response.text}"
        )

    # When the caller needs the created item's body (e.g. to extract item ID),
    # fetch the operation result — status endpoint only returns progress metadata.
    if return_result:
        result_url = f"{location}/result"
        return requests.get(result_url, headers=headers)

    return response

# --------------------------------------------------------
#  Get capacity id function
# --------------------------------------------------------
def get_capacity_id(access_token: str, workspace_id: str) -> str:
    """
    Retrieve the capacity ID assigned to a workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param workspace_id: GUID of the target workspace.
    :returns: Capacity GUID, or None if the workspace has no capacity assigned.
    :raises requests.HTTPError: If the workspace lookup fails.
    """
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    data = response.json()

    # capacity ID shows up as: data["capacityId"]
    return data.get("capacityId")

# --------------------------------------------------------
#  Get Fabric Item Id by display name and type
# --------------------------------------------------------
def get_item_id(access_token: str, item_name: str, item_type: str = None) -> str:
    """
    Retrieve the ID of a Fabric item in the current workspace by display name.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param item_name: Display name of the target item.
    :param item_type: Optional Fabric item type (e.g. "Lakehouse", "Warehouse")
        to disambiguate when multiple items share a display name.
    :returns: Item GUID.
    :raises ValueError: If no matching item is found.
    :raises requests.HTTPError: If the items listing call fails.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    items = response.json().get("value", [])

    for item in items:
        if item["displayName"] == item_name:
            if item_type is None or item["type"] == item_type:
                return item["id"]

    raise ValueError(f"Item '{item_name}' (type={item_type}) not found in workspace.")

# **Operation**

## **Get access token**

In [ ]:
# --------------------------------------------------------
#  Get access token
# --------------------------------------------------------
access_token = get_access_token(tenant_id, client_id, client_secret)

# --------------------------------------------------------
#  Get current Workspace Id
# --------------------------------------------------------
# notebookutils.runtime.context is a dict that exposes the running notebook's
# workspace ID + name, notebook ID + name, default lakehouse ID + name, and
# userId. It is the documented public API for resolving runtime identity.
#
# spark.conf.get("trident.workspace.id") also works but is internal Spark conf
# and is not available in pure-Python notebooks. Reserve spark.conf.* for
# Spark-session tuning (shuffle partitions, AQE, Delta settings).
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
print(f"Workspace Id: {workspace_id}")

# --------------------------------------------------------
#  Get current Capacity Id
# --------------------------------------------------------
capacity_id = get_capacity_id(access_token, workspace_id)
print("Capacity Id:", capacity_id)